In [1]:
import sys
!{sys.executable} -m pip install librosa numpy pandas scikit-learn matplotlib tqdm joblib -q


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


### Exploración del data set

In [2]:
# Cargar y analizar el protocolo de entrenamiento para entender la distribución de clases y ataques
import pandas as pd
import numpy as np
import os

DATASET_DIR  = '/Users/liliana/Downloads/FinalDataset_16khz'
PROTOCOL_PATH = f'{DATASET_DIR}/protocol.txt'

# Cargar protocolo
df = pd.read_csv(PROTOCOL_PATH, sep=' ', header=None,
                 names=['speaker', 'file_name', 'env', 'attack', 'label'])

print(f'Total archivos    : {len(df)}')
print(f'\nDistribución label:')
print(df['label'].value_counts())
print(f'\nSistemas de spoof:')
print(df['attack'].value_counts())


Total archivos    : 80816

Distribución label:
label
spoof       58000
bonafide    22816
Name: count, dtype: int64

Sistemas de spoof:
attack
-              22816
CycleGAN       16000
Diff           16000
StarGAN        16000
TTS             5000
TTS-Diff        2500
TTS-StarGAN     2500
Name: count, dtype: int64


Función para encontrar ruta de cada audio

In [3]:
# Función para encontrar la ruta completa del audio dado su nombre y tipo de ataque
def encontrar_ruta_audio(file_name, attack, dataset_dir):
    """
    Busca la ruta completa del audio dado su nombre y tipo de ataque.
    Real: Dataset/Real/Pais/hablante/archivo.wav
    Spoof: Dataset/Attack/Pais-Pais/hablante-hablante/archivo.wav
    """
    if attack == '-':
        # Es bonafide — buscar en Real/
        for root, dirs, files in os.walk(os.path.join(dataset_dir, 'Real')):
            for f in files:
                if f.startswith(file_name) and f.endswith('.wav'):
                    return os.path.join(root, f)
    else:
        # Es spoof — buscar en la carpeta del attack
        attack_dir = os.path.join(dataset_dir, attack)
        for root, dirs, files in os.walk(attack_dir):
            for f in files:
                if f.startswith(file_name) and f.endswith('.wav'):
                    return os.path.join(root, f)
    return None


Construir índice de rutas

In [4]:
# Construir un índice de archivos para acelerar la búsqueda
from tqdm import tqdm

print('Construyendo índice de archivos...')
indice = {}
for root, dirs, files in os.walk(DATASET_DIR):
    for f in files:
        if f.endswith('.wav'):
            nombre = f.replace('.wav', '')
            indice[nombre] = os.path.join(root, f)

print(f'Archivos indexados: {len(indice)}')

# Verificar que encontramos los archivos del protocolo
df['ruta'] = df['file_name'].map(indice)
encontrados = df['ruta'].notna().sum()
print(f'Archivos del protocolo encontrados: {encontrados}/{len(df)}')


Construyendo índice de archivos...
Archivos indexados: 80816
Archivos del protocolo encontrados: 75816/80816


### ETL

Usaremos el mismo pipeline del ETL_V2

In [5]:
# ETL: Extracción de features utilizando STFT en segmentos uniformes
import librosa
import warnings
import multiprocessing
from joblib import Parallel, delayed

warnings.filterwarnings('ignore', category=UserWarning, module='librosa')

N_FRAMES     = 5
FRAME_LENGTH = 2048
TARGET_SR    = 16000
CORES        = multiprocessing.cpu_count()
OUTPUT_DIR   = '/Users/liliana/Documents/GitHub/Reto_Inteligencia_Artificial_Liliana_Daniele_Alexis/Metricas/ETL_LatinAmerica/'

def calculate_uniform_segments(total_samples, n_frames=5, frame_length=2048):
    segment_size = total_samples // n_frames
    indices = []
    for i in range(n_frames):
        center = i * segment_size + segment_size // 2
        start  = max(0, center - frame_length // 2)
        end    = start + frame_length
        if end > total_samples:
            end   = total_samples
            start = end - frame_length
        indices.append((start, end))
    return indices

def etl_pipeline(file_path, n_frames=5, frame_length=2048, target_sr=16000):
    audio, sr     = librosa.load(file_path, sr=target_sr)
    if len(audio) < frame_length:
        return None
    frame_indices = calculate_uniform_segments(len(audio), n_frames, frame_length)
    features = []
    for start, end in frame_indices:
        stft      = librosa.stft(audio[start:end], n_fft=frame_length, hop_length=frame_length+1)
        features.append(np.abs(stft))
    return np.array(features)

def process_row(row):
    if pd.isna(row['ruta']):
        return None, None
    try:
        features = etl_pipeline(row['ruta'], N_FRAMES, FRAME_LENGTH, TARGET_SR)
        label    = 0 if row['label'] == 'bonafide' else 1
        return features, label
    except Exception:
        return None, None

print('Extrayendo features...')
df_valid = df[df['ruta'].notna()].reset_index(drop=True)
rows     = [row for _, row in df_valid.iterrows()]

results = Parallel(n_jobs=CORES)(
    delayed(process_row)(row)
    for row in tqdm(rows, desc='Procesando audios')
)

X = [r[0] for r in results if r[0] is not None]
y = [r[1] for r in results if r[0] is not None]
X = np.array(X)
y = np.array(y)

print(f'\nShape X: {X.shape}')
print(f'bonafide: {(y==0).sum()}  spoof: {(y==1).sum()}')


Extrayendo features...


Procesando audios: 100%|██████████| 75816/75816 [00:22<00:00, 3297.38it/s]



Shape X: (75816, 5, 1025, 1)
bonafide: 22816  spoof: 53000


Guardar tensores

In [6]:
# Guardar las features y etiquetas para su uso posterior
os.makedirs(OUTPUT_DIR, exist_ok=True)
np.save(os.path.join(OUTPUT_DIR, 'X_fourier_features.npy'), X)
np.save(os.path.join(OUTPUT_DIR, 'y_labels.npy'), y)
print(f'Guardado en {OUTPUT_DIR}')
print(f'X: {X.shape} | y: {y.shape}')


Guardado en /Users/liliana/Documents/GitHub/Reto_Inteligencia_Artificial_Liliana_Daniele_Alexis/Metricas/ETL_LatinAmerica/
X: (75816, 5, 1025, 1) | y: (75816,)
